In [1]:
import huggingface_hub
from datasets import load_dataset
ds = load_dataset("SetFit/sst2")

c:\Users\Udayan\Desktop\udayan\vscode\inference_server\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Repo card metadata block was not found. Setting CardData to empty.


In [2]:
test_data = ds['test']
test_data

Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 1821
})

In [16]:
import torch 
from torch.utils.data import DataLoader
import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
predicted_ans = []
model_path = r"C:\Users\Udayan\Desktop\udayan\vscode\inference_server\models\saved_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()
model.to("cuda")
predicted_ans = []
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)
tokenized_test_data = test_data.map(tokenize_function, batched=True)
tokenized_test_data.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
dataloader = DataLoader(tokenized_test_data, batch_size=32, shuffle=False)
with torch.no_grad():
    for batch in tqdm.tqdm(dataloader):
        input_id = batch["input_ids"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")

        outputs = model(input_ids=input_id, attention_mask=attention_mask)
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        predicted_ans.extend(preds)

100%|██████████| 57/57 [00:04<00:00, 13.86it/s]


In [24]:
from sklearn import metrics
import numpy as np
predicted_ans =list(map(int, predicted_ans))
print(len(predicted_ans))
print(len(test_data['label']))
f1_score = metrics.f1_score(
    y_true=test_data['label'],
    y_pred=predicted_ans,
    average = "binary",
)

1821
1821


In [ ]:
f1_score
##Learning rate was too low im guessing and im too lazy to train it again so whatever let it be


0.8864926220204313